# Generar audio con python

**Curso**: INF3841 - Recuperación de Información   
**Profesor**: Juan Manuel Barrios  
**Fecha**: 31 de agosto de 2026

# 1-Aliasing de señales

El **Teorema de Nyquist-Shannon** dice que si se tiene una onda continua, para digitalizarla correctamente la cantidad de muestras debe ser mayor que el doble de la máxima frecuencia de la onda, es decir para digitalizar una onda de X Hz se necesitan al menos 2 * X + 1 muestras.

Intuitivamente, para poder digitalizar un ciclo de una onda se necesitan al menos dos ejemplos o muestras, ya que se debe poder representar cuando la señal "está arriba" y cuando "está abajo".

Al generar un audio, primero se debe definir el `sample_rate` (cantidad de samples por segundo), que implica la calidad o nivel de resolución del audio. Normalmente se usa un valor como `44100` o `22050`.

El `sample_rate` limita la frecuencia máxima que se puede representar de una onda. Este límite tiene dos efectos:

   - **Efecto 1** (repetición): Si se intenta digitalizar una onda de frecuencia mayor al samplerate (`hertz > sample_rate`), en la práctica se obtendrá una onda de frecuencia `(hertz % sample_rate)` Hz. Por ejemplo, si se usa un samplerate de 10.000 y se desea digitalizar una onda de 12.000 HZ, se obtendrá lo mismo que digitalizar una onda de 2.000 Hz.

  - **Efecto 2** (reflejado): Si se intenta digitalizar una onda de frecuencia mayor a la mitad del samplerate (`hertz > sample_rate / 2`), en la práctica se obtendrá una onda de frecuencia `(sample_rate - hertz)` Hz.  Por ejemplo, si se usa un samplerate de 10.000 y se desea digitalizar una onda de 7.000 HZ, en la práctica será lo mismo que digitalizar una onda de 3.000 Hz.

A este efecto se le denomina **"aliasing"** ya que al digitalizar ondas de frecuencias muy alta (mayor a la mitad del `sample_rate`) se obtiene el mismo resultado que una onda de frecuencia más baja (un "alias").


In [ ]:
import numpy
import matplotlib.pyplot as plt


def mostrar_muestreo(frecuencia_original, samplerate, texto=""):
    # se obtienen varios puntos por segundo para mostrar una curva
    x_tiempo_continuo = numpy.linspace(0, 1, 10000)
    # la tasa de muestreo es igual 1/samplerate
    # se suma un pequeño para epsilon para que aparezca la muestra del segundo 1
    x_tiempo_muestreo = numpy.arange(0, 1.0001, 1 / samplerate)
    # una onda con muestreo continuo y discreto
    y_original = numpy.cos(2 * numpy.pi * frecuencia_original * x_tiempo_continuo)
    y_muestreo = numpy.cos(2 * numpy.pi * frecuencia_original * x_tiempo_muestreo)
    # graficar
    plt.figure(figsize=(14, 3))
    plt.plot(x_tiempo_continuo, y_original, "b-", lw=2, alpha=0.3)
    plt.plot(x_tiempo_muestreo, y_muestreo, "ro")
    plt.ylabel("Amplitud")
    plt.axhline(c="k", lw=0.1)
    plt.axvline(c="k", lw=0.1)
    # se suma un pequeño para epsilon para que aparezca la muestra del segundo 1
    plt.xticks(numpy.arange(0, 1.0001, step=0.1))
    plt.yticks(numpy.arange(-1, 1.0001, step=0.5))
    plt.grid(True, axis='y')
    plt.title("Onda con frecuencia={} Hz y samplerate={}".format(frecuencia_original, samplerate) + texto)
    plt.show()

mostrar_muestreo(frecuencia_original=5, samplerate=10)

## 1.a) Mostrar Efecto 1 de aliasing

Si se intenta generar una onda de frecuencia mayor al samplerate (`hertz > sample_rate`), en la práctica se generá una onda de frecuencia `(hertz % sample_rate)` Hz.

In [ ]:
mostrar_muestreo(frecuencia_original=2,  samplerate=10)
mostrar_muestreo(frecuencia_original=12, samplerate=10, texto=" (mismos puntos que onda de 2 Hz)")
mostrar_muestreo(frecuencia_original=22, samplerate=10, texto=" (mismos puntos que onda de 2 Hz)")


## 1.b) Mostrar Efecto 2 de aliasing

Si se intenta generar una onda de frecuencia mayor a la mitad del samplerate (`hertz > sample_rate / 2`), en la práctica generá una onda de frecuencia `(sample_rate - hertz)` Hz.

In [ ]:
mostrar_muestreo(frecuencia_original=4, samplerate=10)
mostrar_muestreo(frecuencia_original=6, samplerate=10, texto=" (mismos puntos que onda de 4 Hz)")

En resumen, los dos tipos de aliasing se puede resumir en que si una onda de frecuencia `hertz_original` se desea muestrear con tasa `sample_rate` se obtendrá una onda de frecuencia `hertz_real` según la siguiente fórmula:

```
hertz_real = hertz_original % sample_rate
if hertz_real > sample_rate / 2:
    hertz_real = sample_rate - hertz_real
```
 
Por ejemplo:
   * Si se tiene un audio con samplerate de 10.000 y se genera una onda de 29.000 HZ ¿qué frecuencia se oirá en el audio?

   * Si en un audio de 1.000 samples por segundo se genera una onda de 3.700 HZ ¿qué frecuencia se oirá en el audio?  

(Resp: 1.000 Hz y 300 Hz)



# 2- Generar audios como suma de cosenos

A continuación se generarán audios como suma de cosenos de distinta frecuencia. Se generarán usando cosenos con amplitud en el rango `[-1, 1]` que al generar el audio se llevarán al rango de 2 bita `[-32767, 32767]`.

Al sumar ondas es posible superar el rango válido `[-1, 1]`. Hay dos formas para evitar esto:

  * **Normalizar**: Escalar el valor de todos los samples (ajustar el volumen) hasta dejar el máximo y mínimo dentro de [-1, 1]. Podría quedar muy despacio.

  * **Saturar**: La onda se trunca cuando supera 1 o -1. Mantiene las amplitudes originales, pero podría escucharse mal en algunas partes.

### 2.a) funciones auxiliares

In [ ]:
import numpy
import os
import IPython.display as ipd
import matplotlib.pyplot as plt

%matplotlib inline


# genera una onda coseno en el rango [-1, 1]
def generar_onda(hertz, num_samples, sample_rate):
    if hertz > sample_rate / 2:
        hertz_real = hertz % sample_rate  # efecto 1 de aliasing
        if hertz_real > sample_rate / 2:  # efecto 2 de aliasing
            hertz_real = sample_rate - hertz_real
        print(
            " **ATENCION: {} Hz supera el límite {}, en la práctica se oirá una onda de {} Hz".format(
                hertz, sample_rate / 2, hertz_real
            )
        )
    samples = numpy.zeros(num_samples, numpy.float32)
    # este offset evita que todas las ondas comiencen al mismo tiempo en su valor máximo ya que cos(0)=1
    offset = sample_rate / hertz / 4
    for i in range(num_samples):
        samples[i] = (i + offset) * (hertz * 2 * numpy.pi) / sample_rate
    samples = numpy.cos(samples)
    return samples


# Al sumar ondas es posible superar el rango válido [-1, 1], se puede corregir normalizando o saturando.
# Se usará normalización, pero se puede cambiar con normalizar=False para probar como se escucha con saturación.
def generar_audio(largo_segundos, sample_rate, ondas, normalizar=True):
    num_samples = round(largo_segundos * sample_rate)
    audio_samples = numpy.zeros(num_samples, numpy.float32)
    for onda in ondas:
        ciclos_por_segundo, amplitud = onda
        onda_samples = generar_onda(ciclos_por_segundo, num_samples, sample_rate)
        audio_samples += amplitud * onda_samples
    # normalizar al rango [-0.9, 0.9]
    if normalizar:
        amplitud_maxima = 0.9
        sample_maximo = max(numpy.max(audio_samples), abs(numpy.min(audio_samples)))
        if sample_maximo > amplitud_maxima:
            audio_samples *= amplitud_maxima / sample_maximo
    # saturar al rango [-1, 1]
    else:
        audio_samples[audio_samples > 1] = 1
        audio_samples[audio_samples < -1] = -1
    return audio_samples


# funcion que dibuja un array de samples
def mostrar_samples(samples_array, titulo):
    plt.figure(figsize=(25, 5))
    plt.title(titulo, {"fontsize": 32})
    plt.plot(samples_array)
    plt.xlabel("Samples")
    plt.ylabel("Amplitud")
    plt.ylim(top=1, bottom=-1)
    plt.yticks(numpy.arange(-1, 1, step=0.2))
    plt.margins(0)
    plt.show()


def guardar_samples_raw(audio_samples, sample_rate, filename):
    samples_16bits = (audio_samples * 32767).astype(numpy.int16)
    carpeta_temporal = "audios_temporales"
    os.makedirs(carpeta_temporal, exist_ok=True)
    archivo = "{}/{}.{}_{}.raw".format(carpeta_temporal, filename, sample_rate, "s16le")
    print("creando {}".format(archivo))
    samples_16bits.tofile(archivo)

### 2.b) Generar onda con un coseno

In [ ]:
sample_rate = 2000

hertz = 500
num_samples = 3 * sample_rate
samples = generar_onda(hertz, num_samples, sample_rate)

largo_mostrar = int(sample_rate * 0.05)
mostrar_samples(samples[0:largo_mostrar], "onda")

# samples debe estar en el rango [-1, 1]
ipd.Audio(samples, rate=sample_rate, autoplay=True, normalize=False)

### 2.c) Generar onda sumando cosenos

Se suman cosenos de distinta frecuencia y con distinta amplitud.

Se debe ingresar un `sample_rate` del audio generado (22050 o 44100 funciona bien).  


In [ ]:
sample_rate = 44100
largo_onda_segundos = 1

ondas1 = [(300, 1), (1800, 0.25), (4500, 0.25)]
samples1 = generar_audio(largo_onda_segundos, sample_rate, ondas1)

ondas2 = [(600, 0.5), (1900, 1), (2500, 0.25)]
samples2 = generar_audio(largo_onda_segundos, sample_rate, ondas2)

ondas3 = [(100, 0.3), (2400, 1), (5000, 0.25)]
samples3 = generar_audio(largo_onda_segundos, sample_rate, ondas3)

#   en el grafico solo se muestran los primeros 20 milisegundos
largo_mostrar = int(sample_rate * 0.02)

mostrar_samples(samples1[0:largo_mostrar], "ondas1")
mostrar_samples(samples2[0:largo_mostrar], "ondas2")
mostrar_samples(samples3[0:largo_mostrar], "ondas3")

samples = numpy.concatenate((samples1, samples2, samples3))

# samples debe estar en el rango [-1, 1]
ipd.Audio(samples, rate=sample_rate, autoplay=True, normalize=False)

## 2.d) Generar una escala

En este ejemplo se genera una escala de frecuencias, similar a lo que se puede oir en https://www.youtube.com/watch?v=qNf9nzvnd1k

Se incrementarán las frecuencias en forma lineal con un paso fijo (sumar 50Hz cada 0.1 segundos). Esto implica que al principio hay gran cambio (cambia bastante el sonido al cambiar de 100Hz a 150Hz) pero al final no se notará cambio (casi no cambia el sonido entre 10000Hz y 10050Hz)

In [ ]:
def generar_escala(frecuencias, segundos_por_ventana, sample_rate):
    num_ventanas = len(frecuencias)
    samples_por_ventana = round(segundos_por_ventana * sample_rate)
    num_samples = num_ventanas * samples_por_ventana
    audio_samples = numpy.zeros(num_samples, numpy.float32)
    print(
        "sample_rate={} ventanas={} samples_por_ventana={} largo_total={:.1f} segs ({} samples)".format(
            sample_rate,
            num_ventanas,
            samples_por_ventana,
            num_samples / sample_rate,
            num_samples,
        )
    )
    inicio = 0
    for frecuencia in frecuencias:
        # se reusa la función anterior de generar ondas
        onda_samples = generar_onda(frecuencia, samples_por_ventana, sample_rate)
        audio_samples[inicio : inicio + samples_por_ventana] = onda_samples[:]
        inicio += samples_por_ventana
    # no es necesario normalizar ni saturar porque no se estan sumando ondas, siempre estará en [-1,1]
    return audio_samples


segundos_por_ventana = 0.1
sample_rate = 44100

frecuencia_inicio = 100
frecuencia_fin = round(0.8 * sample_rate / 2)
frecuencia_paso = 50

# bajar un poco el volumen global, para que no duelan los oidos :-)
volumen_global = 0.2

# se generan las frecuencias de cada ventana en forma lineal a un paso fijo
frecuencias = range(frecuencia_inicio, frecuencia_fin, frecuencia_paso)

# generar los samples para cada frecuencia
print(
    "generando escala desde {} Hz hasta {} Hz ({} frecuencias, {} segundos cada una)".format(
        frecuencias[0], frecuencias[-1], len(frecuencias), segundos_por_ventana
    )
)
samples = generar_escala(frecuencias, segundos_por_ventana, sample_rate)

# bajar el volumen globalmente
samples *= volumen_global

largo_mostrar = int(sample_rate * 0.02)

inicio_1 = int(sample_rate * 0)
inicio_2 = int(sample_rate * 2)
inicio_3 = int(sample_rate * 4)
inicio_4 = int(sample_rate * 8)
inicio_5 = int(sample_rate * 16)

mostrar_samples(samples[inicio_1 : inicio_1 + largo_mostrar], "frecuencia 1")
mostrar_samples(samples[inicio_2 : inicio_2 + largo_mostrar], "frecuencia 2")
mostrar_samples(samples[inicio_3 : inicio_3 + largo_mostrar], "frecuencia 3")
mostrar_samples(samples[inicio_4 : inicio_4 + largo_mostrar], "frecuencia 4")
mostrar_samples(samples[inicio_5 : inicio_5 + largo_mostrar], "frecuencia 5")

guardar_samples_raw(samples, sample_rate, "escala")
ipd.Audio(samples, rate=sample_rate, autoplay=True, normalize=False)